<a href="https://colab.research.google.com/github/Amw839/Amw839/blob/main/01_extract_cityOfNewyYork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

2a. Attendance from the API, in pages. The API returns at most a set number of rows per request, so you loop with $limit and $offset until a page comes back empty. This is the most common real-world API pattern.

In [ ]:
import requests, pandas as pd, time

URL = "https://data.cityofnewyork.us/resource/xuid-t5nk.json"
PAGE = 50000
pages, offset = [], 0

while True:
    r = requests.get(URL, params={"$limit": PAGE, "$offset": offset, "$order": ":id"}, timeout=60)
    r.raise_for_status()
    batch = r.json()
    if not batch:
        break
    pages.append(pd.DataFrame(batch))
    offset += PAGE
    print(f"Downloaded {offset:,} rows")
    time.sleep(1)  # be polite to the server

attendance = pd.concat(pages, ignore_index=True)
attendance.to_csv("attendance_raw.csv", index=False)
print(attendance.shape)

Downloaded 50,000 rows
Downloaded 100,000 rows
Downloaded 150,000 rows
Downloaded 200,000 rows
Downloaded 250,000 rows
(246196, 10)


2b. Demographics as a CSV download.

In [ ]:
demo = pd.read_csv("https://data.cityofnewyork.us/api/v3/views/nie4-bv6q/export.csv?accessType=DOWNLOAD")
demo.to_csv("demographics_raw.csv", index=False)

2c. School locations as JSON.

In [ ]:
loc = pd.DataFrame(requests.get("https://data.cityofnewyork.us/resource/wg9x-4ke6.json", params={"$limit": 5000}).json())
loc.to_json("locations_raw.json", orient="records")

2d. Income from the Census API. It returns a list of lists, where the first row is the header.

In [ ]:
import requests, pandas as pd
from google.colab import userdata

key = userdata.get("census")
print("Key length:", len(key))

r = requests.get("https://api.census.gov/data/2022/acs/acs5",
                 params={"get": "NAME,B19013_001E",
                         "for": "zip code tabulation area:*",
                         "key": key.strip()},
                 timeout=60)
print("Status:", r.status_code)
print(r.text[:300])

if r.text.strip().startswith("["):
    rows = r.json()
    income = pd.DataFrame(rows[1:], columns=rows[0])
    income.to_csv("income_raw.csv", index=False)
    print("Saved", income.shape)

Key length: 40
Status: 200
[["NAME","B19013_001E","zip code tabulation area"],
["ZCTA5 00601","17526","00601"],
["ZCTA5 00602","20260","00602"],
["ZCTA5 00603","17703","00603"],
["ZCTA5 00606","19603","00606"],
["ZCTA5 00610","22796","00610"],
["ZCTA5 00611","22525","00611"],
["ZCTA5 00612","22305","00612"],
["ZCTA5 00616","2
Saved (33774, 3)


key.strip() removes any hidden space that came along when you copied the key.

What to look for:

“Saved (33…, 3)”: it worked, and income_raw.csv is ready.
“Invalid Key”: activation can take a few minutes. Wait 5 minutes and run again. Also make sure the key in Colab Secrets is the same one from the activation email.
“Missing Key”: the key length will show 0 or an error. Check the secret name CENSUS_KEY and that Notebook access is on.

**Things to watch for when you clean this table in Step 3:**

Some zip codes have an income of -666666666. That’s the Census code for “no data” (areas with very few people). Turn it into a blank, not a real number.
Rename the columns to zip and median_income.
Keep zip as text so 00601 doesn’t turn into 601.

## **Step 3:**
What each part shows you:

.shape gives (rows, columns), for example (33774, 3) means 33,774 rows and 3 columns.
.columns lists every column name.
.head(3) shows the first 3 rows so you can see what the values look like.
dtype=str loads everything as text for now. You’ll fix the types when you clean.

In [ ]:
import pandas as pd

tables = {
    "attendance":   pd.read_csv("attendance_raw.csv", dtype=str),
    "demographics": pd.read_csv("demographics_raw.csv", dtype=str),
    "locations":    pd.read_json("locations_raw.json", dtype=str),
    "income":       pd.read_csv("income_raw.csv", dtype=str),
}

for name, df in tables.items():
    print("=" * 60)
    print(name.upper())
    print("Shape (rows, columns):", df.shape)
    print("Columns:", list(df.columns))
    display(df.head(3))

ATTENDANCE
Shape (rows, columns): (246196, 10)
Columns: ['school', 'schoolyear', 'month_code', 'calmonth', 'gradelevel', 'gradesort', 'roster_count', 'absent', 'present', 'released']


,school,schoolyear,month_code,calmonth,gradelevel,gradesort,roster_count,absent,present,released
0,01M015,2019-20,1,Sep,1,1,26,13,427,0
1,01M015,2019-20,1,Sep,2,2,38,13,620,0
2,01M015,2019-20,1,Sep,3,3,35,22,552,0


DEMOGRAPHICS
Shape (rows, columns): (9096, 39)
Columns: ['DBN', 'School Name', 'Year', 'Total Enrollment', 'Grade 3K+PK (Half Day & Full Day)', 'Grade K', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4', 'Grade 5', 'Grade 6', 'Grade 7', 'Grade 8', 'Grade 9', 'Grade 10', 'Grade 11', 'Grade 12', '# Female', '% Female', '# Male', '% Male', '# Asian', '% Asian', '# Black', '% Black', '# Hispanic', '% Hispanic', '# Multiple Race Categories Not Represented', '% Multiple Race Categories Not Represented', '# White', '% White', '# Students with Disabilities', '% Students with Disabilities', '# English Language Learners', '% English Language Learners', '# Poverty', '% Poverty', 'Economic Need Index']


,DBN,School Name,Year,Total Enrollment,Grade 3K+PK (Half Day & Full Day),Grade K,Grade 1,Grade 2,Grade 3,Grade 4,...,% Multiple Race Categories Not Represented,# White,% White,# Students with Disabilities,% Students with Disabilities,# English Language Learners,% English Language Learners,# Poverty,% Poverty,Economic Need Index
0,01M015,P.S. 015 Roberto Clemente,2015-16,176,14,32,33,39,23,17,...,1.7%,2,1.1%,60,34.1%,16,9.1%,149,84.7%,88.9%
1,01M015,P.S. 015 Roberto Clemente,2016-17,178,17,28,33,27,31,24,...,2.2%,4,2.2%,51,28.7%,12,6.7%,152,85.4%,88.2%
2,01M015,P.S. 015 Roberto Clemente,2017-18,190,17,28,32,33,23,31,...,1.1%,6,3.2%,49,25.8%,8,4.2%,161,84.7%,89.0%


LOCATIONS
Shape (rows, columns): (2190, 41)
Columns: ['fiscal_year', 'system_code', 'location_code', 'location_name', 'beds', 'managed_by_name', 'location_type_description', 'location_category_description', 'grades_text', 'grades_final_text', 'open_date', 'status_descriptions', 'primary_building_code', 'primary_address_line_1', 'state_code', 'x_coordinate', 'y_coordinate', 'longitude', 'latitude', 'community_district', 'council_district', 'census_tract', 'borough_block_lot', 'nta', 'nta_name', 'principal_name', 'principal_title', 'principal_phone_number', 'fax_number', 'geographical_district_code', 'administrative_district_code', 'administrative_district_name', 'community_school_sup_name', 'tier_3_support_location_name', 'tier_3_support_leader_name', 'tier_2_support_location_name', 'highschool_network_location', 'highschool_network_name', 'highschool_network', 'community_district_1', 'police_precinct']


,fiscal_year,system_code,location_code,location_name,beds,managed_by_name,location_type_description,location_category_description,grades_text,grades_final_text,...,administrative_district_name,community_school_sup_name,tier_3_support_location_name,tier_3_support_leader_name,tier_2_support_location_name,highschool_network_location,highschool_network_name,highschool_network,community_district_1,police_precinct
0,2020,15K001,K001,P.S. 001 The Bergen,331500010001,DOE,General Academic,Elementary,"PK,0K,01,02,03,04,05,SE","PK,0K,01,02,03,04,05",...,COMMUNITY SCHOOL DISTRICT 15,"SKOP, ANITA",NYCDOE Borough Office - Brooklyn North,NULL,School Support Team 5- Brooklyn North,NULL,NULL,NULL,307,72
1,2020,17K002,K002,Parkside Preparatory Academy,331700010002,DOE,General Academic,Junior High-Intermediate-Middle,"06,07,08,SE","06,07,08",...,COMMUNITY SCHOOL DISTRICT 17,"ELLIS, CLARENCE",NYCDOE Borough Office - Brooklyn South,Mauriciere de Govia,School Support Team 2- Brooklyn South,NULL,NULL,NULL,309,71
2,2020,13K003,K003,P.S. 003 The Bedford Village,331300010003,DOE,General Academic,Elementary,"PK,0K,01,02,03,04,05,SE","PK,0K,01,02,03,04,05",...,COMMUNITY SCHOOL DISTRICT 13,"SAMUELS, KAMAR",NYCDOE Borough Office - Brooklyn North,NULL,School Support Team 3- Brooklyn North,NULL,NULL,NULL,303,79


INCOME
Shape (rows, columns): (33774, 3)
Columns: ['NAME', 'B19013_001E', 'zip code tabulation area']


,NAME,B19013_001E,zip code tabulation area
0,ZCTA5 00601,17526,00601
1,ZCTA5 00602,20260,00602
2,ZCTA5 00603,17703,00603


How to find the DBN column: a DBN is the school ID. It looks like 01M015: two digits, a letter for the borough (M, X, K, Q or R), then three digits. In each table’s .head(3), look for the column with values in that pattern. It may be called dbn, DBN, or something else. The income table won’t have one; it joins on zip code instead.

Then add a Text cell under the output with your notes, like this (fill in what you find):

   Raw data notes
   - attendance:   DBN column = dbn
   - demographics: DBN column = dbn
   - locations:    DBN column = ..., zip column = ...
   - income:       no DBN; zip column = "zip code tabulation area"